# 02 – Build the G50T dataset (exam step 2: reasoning traces)

Turns your recorded solutions (`solutions/g50t_solutions.json`) into a training/evaluation dataset.

Two kinds of rows:
* **play** – the board at each step of your solution, the move you made, and a reasoning text for it.
* **explore** – from the same positions, *every* action is tried on a copy of the game and the effect is recorded ("DOWN: blocked by a wall"). This teaches how the game reacts.

Every row has: screenshot, board as text, level, step, action, reasoning, `split` (levels 1–5 = **test**, 6–7 = **train**).

**Part 1** (fast, no AI) builds everything with rule-based reasoning → already a complete dataset.
**Part 2** (optional) lets `lisa-pro` rewrite the reasoning in natural language. Run it if time allows.
Run in the same folder as notebook 01.

In [ ]:
%pip install -q "arc-agi==0.9.9" pillow datasets pyarrow openai

In [ ]:
import copy, io, json, random
from pathlib import Path
import numpy as np
import arc_agi
from arc_agi import OperationMode
from arc_agi.rendering import frame_to_rgb_array
from PIL import Image
from IPython.display import display

ENV_DIR = "./environment_files"
GAME_ID = "g50t"
OUT = Path("dataset"); OUT.mkdir(exist_ok=True)
solutions = json.loads(Path("solutions/g50t_solutions.json").read_text())
print("levels with a solution:", {int(k) + 1: len(v) for k, v in solutions.items()})

arcade = arc_agi.Arcade(operation_mode=OperationMode.OFFLINE, environments_dir=ENV_DIR)
def new_env():
    env = arcade.make(GAME_ID); return env, env.reset()
def act(env, name):
    return env.step({a.name: a for a in env.action_space}[name])
def render(obs, scale=8):
    return Image.fromarray(frame_to_rgb_array(steps=0, frame=obs.frame[-1], scale=scale))

## Helpers: read the hidden game state, board as text

In [ ]:
def find_attr_path(obj, target="vgwycxsxjz", depth=0, path=()):
    if hasattr(obj, target): return path
    if depth >= 3 or not hasattr(obj, "__dict__"): return None
    for name, value in vars(obj).items():
        if value is None or isinstance(value, (int, float, str, bytes, list, dict, tuple, np.ndarray)): continue
        found = find_attr_path(value, target, depth + 1, path + (name,))
        if found is not None: return found
    return None

_env, _ = new_env()
GAME_PATH = find_attr_path(_env)
def world(e):
    g = e
    for name in GAME_PATH: g = getattr(g, name)
    return g.vgwycxsxjz

CELL = 6   # one move = 6 pixels
def player_cell(e):
    p = world(e).dzxunlkwxt
    return ((p.x - 1) // CELL, (p.y - 1) // CELL)          # (column, row) on the move grid
def n_ghosts(e):
    return len(world(e).rloltuowth)

COLOR_CHARS = "0123456789abcdef"
def board_text(obs):
    f = np.asarray(obs.frame[-1])
    return "\n".join("".join(COLOR_CHARS[int(v) % 16] for v in row) for row in f)

def png_bytes(img):
    buf = io.BytesIO(); img.save(buf, format="PNG"); return buf.getvalue()

NAMES = {"ACTION1": "UP", "ACTION2": "DOWN", "ACTION3": "LEFT", "ACTION4": "RIGHT", "ACTION5": "REWIND"}
print("game object found at:", GAME_PATH, "| start cell:", player_cell(_env))

## Your level notes (fill these in!)
One or two sentences per level: what the trick is. They are used in the reasoning texts. Better notes → better data.

In [ ]:
LEVEL_NOTES = {
    1: "The red gate only stays open while something stands on the red button. Walk to the button, REWIND so the ghost repeats the walk and holds the button, then go through the gate to the goal.",
    2: "Two buttons and two gates. Use two rewinds so ghosts hold the buttons while I pass.",
    3: "",   # <- your notes
    4: "",
    5: "",
    6: "",
    7: "",
}

## What does one action do? (tried on a copy of the game)

In [ ]:
def try_action(env, name):
    """Apply an action to a COPY and describe the effect. Returns (description, obs_after, effect_code)."""
    e = copy.deepcopy(env)
    lvl0, cell0, g0 = None, player_cell(env), n_ghosts(env)
    o_before_lvl = world(env)  # just to keep a reference
    o = act(e, name)
    if o is None:
        return "nothing happened", None, "none"
    if o.state.name == "GAME_OVER":
        return f"{NAMES[name]}: GAME OVER (died or ran out of moves)", o, "game_over"
    try:
        cell1, g1 = player_cell(e), n_ghosts(e)
    except Exception:
        cell1, g1 = None, None
    if name == "ACTION5":
        if g1 is not None and g1 > g0:
            return f"REWIND: I jump back to the start and my previous path becomes ghost #{g1}", o, "rewind"
        return "REWIND: the timeline was reset", o, "rewind"
    if cell1 == cell0:
        return f"{NAMES[name]}: blocked, I stay at {cell0}", o, "blocked"
    return f"{NAMES[name]}: I move from {cell0} to {cell1}", o, "moved"

## Part 1 – build all rows

In [ ]:
TEST_LEVELS = {1, 2, 3, 4, 5}       # course: evaluate on levels 1-5 -> never train on them
rows = []
env, obs = new_env()
for lvl_idx in range(7):
    key = str(lvl_idx)
    if key not in solutions:
        print(f"level {lvl_idx+1}: no solution recorded -> stopping here"); break
    level = lvl_idx + 1
    split = "test" if level in TEST_LEVELS else "train"
    moves = solutions[key]
    history = []
    for step, expert in enumerate(moves):
        img = render(obs)
        effects = {a: try_action(env, a) for a in NAMES}
        works = [NAMES[a] for a, (_, _, c) in effects.items() if c in ("moved", "rewind")]
        blocked = [NAMES[a] for a, (_, _, c) in effects.items() if c == "blocked"]
        cell, ghosts = player_cell(env), n_ghosts(env)

        # ---- exploration rows: every action from this position ----
        for a, (desc, o_after, code_) in effects.items():
            if o_after is None: continue
            rows.append(dict(kind="explore", level=level, step=step, split=split,
                image=png_bytes(img), image_after=png_bytes(render(o_after)),
                board_text=board_text(obs), action=a,
                question=f"You are playing G50T. You press {a} ({NAMES[a]}). Look at the board before and after. What happened?",
                answer=desc))

        # ---- play row: the move you actually made ----
        desc, _, _ = effects[expert]
        reasoning = (f"I am at {cell} with {ghosts} ghost(s) replaying. "
                     f"Moves that work here: {', '.join(works) or 'none'}; blocked: {', '.join(blocked) or 'none'}. "
                     + (f"Plan for this level: {LEVEL_NOTES.get(level)} " if LEVEL_NOTES.get(level) else "")
                     + f"I choose {NAMES[expert]} -> {desc.split(': ', 1)[-1]}.")
        rows.append(dict(kind="play", level=level, step=step, split=split,
            image=png_bytes(img), image_after=None, board_text=board_text(obs), action=expert,
            question=("You are playing G50T. Actions: ACTION1=up, ACTION2=down, ACTION3=left, ACTION4=right, ACTION5=rewind. "
                      f"Your last actions: {history[-5:]}. Look at the board and choose the next action. "
                      "Explain briefly, then end with 'Action: ACTIONn'."),
            answer=reasoning + f"\nAction: {expert}"))

        obs = act(env, expert); history.append(NAMES[expert])
    print(f"level {level}: {len(moves)} moves, split={split}, rows so far: {len(rows)}")

kinds = {k: sum(r['kind'] == k for r in rows) for k in ('play', 'explore')}
print("\nTOTAL rows:", len(rows), kinds, "| images:", sum(1 for r in rows if r['image']) + sum(1 for r in rows if r['image_after']))

### Look at a few rows

In [ ]:
for r in random.sample([r for r in rows if r["kind"] == "play"], 2) + random.sample([r for r in rows if r["kind"] == "explore"], 2):
    print(f"--- {r['kind']} | level {r['level']} step {r['step']} | {r['action']}")
    print("Q:", r["question"]); print("A:", r["answer"])
    display(Image.open(io.BytesIO(r["image"])).resize((256, 256)))

## Save as Parquet (the file you upload to Moodle)

In [ ]:
from datasets import Dataset, Features, Value, Image as HFImage

def to_hf(rows):
    data = {k: [r[k] for r in rows] for k in rows[0]}
    data["image"] = [{"bytes": b, "path": None} for b in data["image"]]
    data["image_after"] = [{"bytes": b, "path": None} if b else None for b in data["image_after"]]
    feats = Features({"kind": Value("string"), "level": Value("int32"), "step": Value("int32"), "split": Value("string"),
                      "image": HFImage(), "image_after": HFImage(), "board_text": Value("string"),
                      "action": Value("string"), "question": Value("string"), "answer": Value("string")})
    return Dataset.from_dict(data, features=feats)

ds = to_hf(rows)
ds.to_parquet(OUT / "g50t_dataset.parquet")
print(ds)
print("file size:", round((OUT / "g50t_dataset.parquet").stat().st_size / 1e6, 1), "MB")
print("split counts:", {s: sum(1 for r in rows if r['split'] == s) for s in ('train', 'test')})

## Part 2 (optional) – natural-language reasoning with lisa-pro
Rewrites the `play` answers into real reasoning traces. Needs `api_key.txt` (your LISA key from Lab 1/Ex3).
Progress is cached in `dataset/llm_cache.json`, so you can stop and resume. Run Part 1's save cell again afterwards? **No** – the last cell here saves a second file `g50t_dataset_llm.parquet`.

In [ ]:
from openai import OpenAI
import base64, time
client = OpenAI(base_url="https://chat-1.ki-awz.iisys.de/api", api_key=Path("api_key.txt").read_text().strip())
MODEL = "lisa-pro"
cache_file = OUT / "llm_cache.json"
cache = json.loads(cache_file.read_text()) if cache_file.exists() else {}

def uri(b): return "data:image/png;base64," + base64.b64encode(b).decode()

def llm_reasoning(r):
    facts = r["answer"].split("\nAction:")[0]
    prompt = (f"You are writing a short reasoning trace for a player of the grid puzzle game G50T.\n"
              f"Facts about this moment (true, use them): {facts}\n"
              f"The player chose {r['action']} ({NAMES[r['action']]}).\n"
              "Write 2-4 sentences in first person explaining what you see on the board and why this move makes sense "
              "for reaching the goal. Do not invent rules that are not in the facts. Do not mention 'facts'.")
    resp = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": [
        {"type": "text", "text": prompt}, {"type": "image_url", "image_url": {"url": uri(r["image"])}}]}])
    return resp.choices[0].message.content.strip()

play_rows = [r for r in rows if r["kind"] == "play"]
for i, r in enumerate(play_rows):
    key = f"{r['level']}-{r['step']}"
    if key in cache: continue
    try:
        cache[key] = llm_reasoning(r)
    except Exception as ex:
        print("error at", key, repr(ex)); time.sleep(5); continue
    if i % 10 == 0:
        cache_file.write_text(json.dumps(cache, indent=1)); print(f"{len(cache)}/{len(play_rows)} done")
cache_file.write_text(json.dumps(cache, indent=1))
print("finished:", len(cache), "/", len(play_rows))

In [ ]:
rows_llm = []
for r in rows:
    r2 = dict(r)
    key = f"{r['level']}-{r['step']}"
    if r["kind"] == "play" and key in cache:
        r2["answer"] = cache[key] + f"\nAction: {r['action']}"
    rows_llm.append(r2)
to_hf(rows_llm).to_parquet(OUT / "g50t_dataset_llm.parquet")
print("saved dataset/g50t_dataset_llm.parquet")
print(random.choice([r for r in rows_llm if r["kind"] == "play"])["answer"])